# RAD + SkinCAP — chạy huấn luyện đầy đủ trên Google Colab

Notebook này cấu hình một thí nghiệm SkinCAP 30 epoch có thể khôi phục an toàn sau khi Colab ngắt:

1. Chuẩn bị repository và dependency.
2. Kiểm tra GPU, Google Drive, ClinicalBERT và dữ liệu SkinCAP.
3. Xác thực một batch dữ liệu thật.
4. Chọn một thư mục chạy có timestamp và tạo cấu hình 30 epoch.
5. Khởi chạy hoặc tiếp tục đúng thư mục chạy đó.
6. Kiểm tra checkpoint, metrics và log đã lưu trên Drive.

**Trước khi chạy:** chọn **Runtime → Change runtime type → GPU**. Không chạy lại tiền xử lý SkinCAP và không chạy `skin.sh`/Slurm scripts.

> Mỗi thí nghiệm mới nằm trong `RAD-outputs/skin-full-30ep/YYYYMMDD_HHMMSS`. Muốn resume, giữ nguyên `RUN_ID` đã in ra khi tạo lần chạy đầu.

## 1. Chuẩn bị Python và repository

Ô này clone repository nếu chưa có. Nếu đã có từ phiên Colab hiện tại, nó chỉ cập nhật metadata Git; không xóa output trên Drive.

In [ ]:
import sys
from pathlib import Path

print("Python:", sys.version)

REPO_DIR = Path("/content/RAD-colab")
REPO_URL = "https://github.com/trisle0805/RAD-colab.git"

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} remote -v
    print(f"✅ Đã có repository: {REPO_DIR}")

## 2. Cài dependency và kiểm tra GPU

`requirements-colab-skin.txt` không cài lại `torch`/`torchvision`, để giữ cặp CUDA tương thích mà Colab đang cung cấp.

In [ ]:
%cd /content/RAD-colab
%pip install -q -r requirements-colab-skin.txt

import importlib
from importlib.metadata import version

modules = {
    "yaml": "PyYAML",
    "numpy": "numpy",
    "pandas": "pandas",
    "PIL": "Pillow",
    "tqdm": "tqdm",
    "cv2": "opencv-python-headless",
    "sklearn": "scikit-learn",
    "scipy": "scipy",
    "skimage": "scikit-image",
    "pydicom": "pydicom",
    "nibabel": "nibabel",
    "tensorboardX": "tensorboardX",
    "transformers": "transformers",
    "timm": "timm",
    "einops": "einops",
    "peft": "peft",
}

for module, package in modules.items():
    importlib.import_module(module)
    print(f"✅ {module:15} {version(package)}")

import torch
import torchvision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA:", torch.version.cuda)
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime → Change runtime type → GPU"
print("GPU:", torch.cuda.get_device_name(0))

## 3. Mount Drive, khai báo đường dẫn riêng và chọn lần chạy

Chỉ sửa các đường dẫn riêng nếu vị trí dữ liệu, ClinicalBERT hoặc output trên Drive thay đổi.

- Để `RUN_ID = None` khi chủ động bắt đầu một thí nghiệm mới. Notebook tạo và in một timestamp dạng `YYYYMMDD_HHMMSS`.
- Khi Colab ngắt hoặc reset runtime, dán timestamp đã in vào `RUN_ID` để tiếp tục **đúng** lần chạy đó.
- Không để `RUN_ID = None` khi muốn resume, vì điều này sẽ tạo thí nghiệm mới.

In [ ]:
from datetime import datetime, timezone
from google.colab import drive
from pathlib import Path

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

SKINCAP_ROOT = Path("/content/drive/MyDrive/Cao học/Experiments/SkinCAP")
TRAIN_CSV = SKINCAP_ROOT / "skincap_50_train_set.csv"
TEST_CSV = SKINCAP_ROOT / "skincap_50_test_set.csv"
IMAGE_ROOT = SKINCAP_ROOT / "skincap"
BERT_DIR = Path("/content/models/Bio_ClinicalBERT")

EXPERIMENT_NAME = "skin-full-30ep"
RUNS_ROOT = Path("/content/drive/MyDrive/Cao học/RAD-outputs") / EXPERIMENT_NAME
RUN_ID = None  # Thí nghiệm mới. Resume: đặt ví dụ "20261002_143015".

for label, path, check in [
    ("Train CSV", TRAIN_CSV, Path.is_file),
    ("Test CSV", TEST_CSV, Path.is_file),
    ("Thư mục ảnh", IMAGE_ROOT, Path.is_dir),
    ("ClinicalBERT", BERT_DIR, Path.is_dir),
]:
    ok = check(path)
    print(f"{'✅' if ok else '❌'} {label}: {path}")
    assert ok, f"Không tìm thấy {label}: {path}"

png_count = len(list(IMAGE_ROOT.glob("*.png")))
print("Số ảnh PNG:", png_count)
assert png_count > 0, "Không có ảnh PNG trong IMAGE_ROOT."

RUNS_ROOT.mkdir(parents=True, exist_ok=True)
if RUN_ID is None:
    RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    RUN_DIR = RUNS_ROOT / RUN_ID
    RUN_DIR.mkdir(parents=False, exist_ok=False)
    print(f"✅ Tạo lần chạy mới: {RUN_ID} (UTC)")
    print("⚠️ Lưu RUN_ID này. Dán lại vào biến RUN_ID để resume sau khi runtime reset.")
else:
    RUN_DIR = RUNS_ROOT / RUN_ID
    latest_checkpoint = RUN_DIR / "checkpoints" / "latest.pt"
    assert latest_checkpoint.is_file(), (
        f"Không thể resume: không tìm thấy checkpoint hoàn chỉnh {latest_checkpoint}"
    )
    print(f"✅ Resume lần chạy: {RUN_ID}")

for directory in ["checkpoints", "metrics", "predictions", "tensorboard", "logs"]:
    (RUN_DIR / directory).mkdir(exist_ok=True)

print("Run directory:", RUN_DIR)

## 4. Kiểm tra ClinicalBERT và một batch SkinCAP

Ô này không train. Nó xác nhận model BERT cục bộ có hidden size 768 và loader đọc đúng CSV, ảnh và 50 nhãn trước khi bắt đầu thí nghiệm dài.

In [ ]:
%cd /content/RAD-colab

from transformers import AutoConfig, AutoTokenizer
from torch.utils.data import DataLoader
from dataset.dataset_entity import Skin_Train_Dataset
from dataset.test_dataset import Skin_Test_Dataset

bert_config = AutoConfig.from_pretrained(BERT_DIR, local_files_only=True)
AutoTokenizer.from_pretrained(BERT_DIR, local_files_only=True)
assert bert_config.hidden_size == 768, (
    f"ClinicalBERT phải có hidden_size=768, nhận được {bert_config.hidden_size}."
)
print("✅ ClinicalBERT:", bert_config.model_type, "hidden_size=", bert_config.hidden_size)

train_dataset = Skin_Train_Dataset(TRAIN_CSV, image_res=512, image_root=IMAGE_ROOT)
test_dataset = Skin_Test_Dataset(TEST_CSV, image_res=512, image_root=IMAGE_ROOT)
batch = next(iter(DataLoader(train_dataset, batch_size=2, num_workers=0)))

print("Số mẫu train:", len(train_dataset))
print("Số mẫu test :", len(test_dataset))
print("Batch image :", tuple(batch["image"].shape))
print("Batch label :", tuple(batch["label"].shape))
assert tuple(batch["image"].shape) == (2, 3, 512, 512)
assert tuple(batch["label"].shape) == (2, 50)
print("✅ Dữ liệu SkinCAP sẵn sàng.")

## 5. Tạo cấu hình huấn luyện đầy đủ

Giữ `batch_size=4` và `grad_accumulation_steps=4` vì đây là cấu hình smoke test đã chạy thành công. Cấu hình chạy 30 epoch; validation và lưu trạng thái chạy sau từng epoch hoàn chỉnh.

In [ ]:
import yaml

with open("configs/skin.yaml", encoding="utf-8") as file:
    config = yaml.safe_load(file)

config.update({
    "batch_size": 4,
    "test_batch_size": 4,
    "num_workers": 2,
    "test_num_workers": 2,
    "grad_accumulation_steps": 4,
})
config["schedular"].update({
    "epochs": 30,
    "warmup_epochs": 20,
})

FULL_CONFIG = RUN_DIR / "skin-full-30ep.yaml"
FULL_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
print(FULL_CONFIG.read_text(encoding="utf-8"))

## 6. Khởi chạy hoặc resume an toàn

Chạy ô này để bắt đầu hoặc tiếp tục lần chạy đã chọn ở bước 3.

Sau mỗi **epoch hoàn chỉnh** (train + validation + metrics), source sẽ lưu trên Drive:

- `checkpoints/latest.pt`: checkpoint mới nhất, ghi đè an toàn để resume từ epoch kế tiếp.
- `checkpoints/epoch_010.pt`, `epoch_020.pt`, `epoch_030.pt`: checkpoint mốc.
- `metrics/metrics_history.csv` và `metrics/latest_metrics.json`: lịch sử và metrics mới nhất.
- `predictions/epoch_XXX.npz`: dữ liệu prediction nén cho mỗi epoch.
- `logs/training.log` và `tensorboard/`: log bền vững.

Nếu Colab ngắt giữa một epoch, chỉ trạng thái của epoch hoàn chỉnh gần nhất được dùng để resume. Không tạo `RUN_ID` mới khi resume.

In [ ]:
latest_checkpoint = RUN_DIR / "checkpoints" / "latest.pt"
if latest_checkpoint.is_file():
    print("✅ Sẽ resume từ checkpoint hoàn chỉnh:", latest_checkpoint)
else:
    print("ℹ️ Chưa có checkpoint: thí nghiệm sẽ bắt đầu từ epoch 1.")

%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
%cd /content/RAD-colab

!python main_rad.py \
  --config {FULL_CONFIG} \
  --dataset skin \
  --train_csv {TRAIN_CSV} \
  --test_csv {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --output_dir {RUN_DIR} \
  --bert_model_name {BERT_DIR} \
  --guideline_path ./guideline/qwen_maxtoken2k_skincap50_4sources.jsonl \
  --max_length 512 \
  --embed_dim 768 \
  --loss_ratio 0.1 \
  --contrast_ratio_text 0.1 \
  --contrast_ratio_vision 0.001

## 7. Kiểm tra trạng thái đã lưu trên Drive

Sau khi một epoch hoàn thành, chạy ô này để xem checkpoint có thể resume, metrics mới nhất và phần cuối của log. Ô này chỉ đọc file; không bắt đầu huấn luyện.

In [ ]:
import json
import pandas as pd

latest_checkpoint = RUN_DIR / "checkpoints" / "latest.pt"
history_path = RUN_DIR / "metrics" / "metrics_history.csv"
latest_metrics_path = RUN_DIR / "metrics" / "latest_metrics.json"
log_path = RUN_DIR / "logs" / "training.log"

print("Run directory:", RUN_DIR)
print("Latest checkpoint:", latest_checkpoint if latest_checkpoint.is_file() else "chưa có")
print("Milestone checkpoints:", [path.name for path in sorted((RUN_DIR / "checkpoints").glob("epoch_*.pt"))])

if history_path.is_file():
    history = pd.read_csv(history_path)
    display(history.tail(5))
else:
    print("ℹ️ Chưa có metrics_history.csv; epoch đầu tiên chưa hoàn thành.")

if latest_metrics_path.is_file():
    print("\nLatest metrics:")
    print(json.dumps(json.loads(latest_metrics_path.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))

if log_path.is_file():
    print("\n20 dòng log cuối:")
    print("\n".join(log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-20:]))